# Agent MCP 工具接入

预计阅读时间：15-20 分钟。

本节在 `03_agent.py` 的基础上只增加一件事：**MCP 工具接入**。

前面已经有了：

- Agent Loop：模型调用工具，程序执行工具，再把结果反馈给模型；
- Bash 工具：让模型能操作当前环境；
- Skill：按需加载可复用经验。

MCP 解决的是另一个问题：**如何用标准方式接入外部工具**。例如文档搜索、部署系统、Issue 系统、数据库查询等能力，不一定都要写进 Agent 主程序里。


## 参考资料

本节参考本地 `reference/learn-claude-code/s19_mcp_plugin/README.md`。

原文中的核心说法是：MCP 定义了 Agent 如何发现和调用外部工具。教学版用 mock handler 模拟外部 server，真实版通常会通过 stdio、HTTP、SSE 或 WebSocket 等方式通信。


## MCP 要解决什么问题

如果没有 MCP，Agent 的每个工具都要手写在本地代码里：

- 工具定义写在 Agent 中；
- 参数格式写在 Agent 中；
- 执行逻辑也写在 Agent 中。

当工具来自外部服务时，这种方式很难维护。MCP 的思路是让外部服务自己声明工具，Agent 通过统一协议发现工具并调用工具。


## MCP 的几个核心概念

| 概念 | 作用 |
| --- | --- |
| MCP Client | Agent 端客户端，负责连接 server、发现工具、调用工具 |
| MCP Server | 外部服务，提供工具列表和工具执行能力 |
| Tool Discovery | 发现 server 暴露了哪些工具 |
| Tool Call | 按工具名和参数调用外部工具 |
| Tool Pool | Agent 当前能使用的全部工具集合 |
| `mcp__server__tool` | 给 MCP 工具加命名空间，避免不同 server 的工具名冲突 |

`04_agent.py` 中的 MCP server 是教学用 mock，不需要真的启动外部服务。


## 1. 导入 04_agent.py

`04_agent.py` 包含前面已有的基础能力，也包含本节新增的 MCP 相关函数。

如果某个函数在 `03_agent.py` 和 `04_agent.py` 中都有，本节直接从 `04_agent.py` 导入使用，避免重复维护两份代码。


In [ ]:
import importlib
import sys
from pathlib import Path

AGENT_DIR = Path("books/06_Agent") if Path("books/06_Agent").exists() else Path.cwd()
if str(AGENT_DIR) not in sys.path:
    sys.path.insert(0, str(AGENT_DIR))

agent = importlib.import_module("04_agent")

# 为了让 notebook 从头执行时结果稳定，先清空已经连接的 mock MCP server。
agent.mcp_clients.clear()

print("Agent dir:", AGENT_DIR)
print("History path:", agent.HISTORY_PATH)


## 2. 先看内置工具池

在连接 MCP server 之前，工具池里只有内置工具。

`04_agent.py` 中的内置工具包括：

- `bash`：执行 shell 命令；
- `load_skill`：加载 Skill 完整说明；
- `connect_mcp`：连接 MCP server。


In [ ]:
tools, handlers = agent.assemble_tool_pool()

print("tool names:")
for tool in tools:
    print("-", tool["name"])

print("\nhandler names:", sorted(handlers))


## 3. MCPClient：教学版客户端

`MCPClient` 模拟真实 MCP Client 的两个关键动作：

1. `register`：模拟工具发现，把 server 提供的工具定义和处理函数注册进来；
2. `call_tool`：模拟工具调用，根据工具名和参数执行对应 handler。

真实 MCP 会通过协议和外部进程或服务通信。本节为了教学，把外部 server 简化成本地 Python 函数。


In [ ]:
client = agent.MCPClient("demo")
client.register(
    tool_defs=[{
        "name": "echo",
        "description": "Echo input text. (readOnly)",
        "inputSchema": {
            "type": "object",
            "properties": {"text": {"type": "string"}},
            "required": ["text"],
        },
    }],
    handlers={"echo": lambda text: f"echo: {text}"},
)

print(client.tools)
print(client.call_tool("echo", {"text": "hello mcp"}))


## 4. 连接 docs MCP server

`04_agent.py` 提供了两个 mock MCP server：

- `docs`：模拟文档服务；
- `deploy`：模拟部署服务。

先连接 `docs`。连接后，`docs` server 暴露的工具会被记录在 `agent.mcp_clients` 中。


In [ ]:
print(agent.connect_mcp("docs"))
print("\nconnected servers:")
print(agent.list_mcp_servers())


## 5. 连接后重新组装工具池

连接 MCP server 后，工具池发生变化。

`assemble_tool_pool()` 会把内置工具和 MCP 工具合并到同一个工具列表中，并为 MCP 工具生成带命名空间的名字。


In [ ]:
tools, handlers = agent.assemble_tool_pool()

for tool in tools:
    print(f"{tool['name']}: {tool['description']}")


## 6. 为什么 MCP 工具名要加前缀

不同 server 可能暴露同名工具。例如两个服务都可能有 `search`。

因此 MCP 工具统一命名为：

```text
mcp__{server}__{tool}
```

例如 `docs` server 的 `search` 工具会变成：

```text
mcp__docs__search
```

这样可以避免工具名冲突。


In [ ]:
for name in ["docs", "my docs", "deploy/v1", "server.name"]:
    print(name, "->", agent.normalize_mcp_name(name))


## 7. 直接调用 MCP 工具

工具池里除了 `tools`，还有对应的 `handlers`。模型看到的是工具定义；程序执行时用 `handlers` 找到真正的处理函数。

下面直接调用 `mcp__docs__search`，等价于模型选择了这个 MCP 工具。


In [ ]:
tools, handlers = agent.assemble_tool_pool()

print(handlers["mcp__docs__search"](query="Agent Harness"))
print(handlers["mcp__docs__get_version"]())


## 8. 再连接 deploy MCP server

MCP 的优势是外部能力可以继续扩展。连接 `deploy` 后，工具池中会同时出现 `docs` 和 `deploy` 的工具。


In [ ]:
print(agent.connect_mcp("deploy"))
print("\nconnected servers:")
print(agent.list_mcp_servers())

print("\nall tools:")
tools, handlers = agent.assemble_tool_pool()
for tool in tools:
    print("-", tool["name"])


## 9. 调用 deploy 工具

`deploy` server 在教学版中只提供一个只读工具：`status`。

真实系统中，部署相关工具可能包含高风险操作，例如触发发布、回滚服务、修改配置。这类工具通常需要和 Permission、Hooks 结合使用。


In [ ]:
print(handlers["mcp__deploy__status"](service="course-site"))


## 10. System Prompt 会显示 MCP 状态

`build_system()` 会把当前已连接的 MCP server 写入 system prompt。

这样模型知道：

- 可以通过 `connect_mcp(name)` 连接 server；
- 已经有哪些 server 连接；
- 连接后工具名采用 `mcp__server__tool` 格式。


In [ ]:
print(agent.build_system())


## 11. 在 Agent Loop 中使用 MCP

完整运行时，模型并不是直接调用 Python 函数，而是通过工具调用 block 请求使用工具。

大致流程是：

```text
用户请求
  -> 模型调用 connect_mcp
  -> 程序连接 server 并返回发现结果
  -> 模型看到新工具
  -> 模型调用 mcp__docs__search 等 MCP 工具
  -> 程序执行并返回结果
  -> 模型给出最终回答
```

下面的 cell 需要配置好 `.env` 中的模型 Key 才能运行。


In [ ]:
messages = [{
    "role": "user",
    "content": "连接 docs MCP server，搜索 Agent Harness，并总结搜索结果。",
}]

# 运行时会真实调用模型。如果只是阅读本节，可以跳过这个 cell。
# agent.agent_loop(messages)
# agent.save_history(messages)
# messages[-1]["content"]


## 12. 与真实 MCP 的区别

本节的实现是教学版 mock，重点是理解结构，不是实现完整协议。

| 项目 | 教学版 `04_agent.py` | 真实 MCP |
| --- | --- | --- |
| Server | Python 函数模拟 | 外部进程或远程服务 |
| 工具发现 | `register(tool_defs, handlers)` | `tools/list` |
| 工具调用 | `call_tool()` 调本地 handler | `tools/call` |
| 通信方式 | 无真实通信 | stdio、HTTP、SSE、WebSocket 等 |
| 权限标注 | 写在 description 中 | 可以有结构化 annotations |

教学版保留了最核心的思想：发现工具、组装工具池、调用外部工具。


## 小结

MCP 让 Agent 可以用标准方式接入外部工具。

本节最重要的几个点是：

- MCP server 负责提供工具；
- MCP client 负责发现和调用工具；
- Harness 把内置工具和 MCP 工具组装成统一工具池；
- MCP 工具使用 `mcp__server__tool` 命名，避免冲突；
- 工具池变化后，Agent 需要让模型看到新的工具列表。

在完整 Agent Harness 中，MCP 通常会和 Permission、Hooks、System Prompt 动态组装一起使用。
